# JN1 · Getting the data in the door

Every housing number Berkeley reports to the state begins as a building permit — a row in a spreadsheet someone at the city filled in by hand. Before we can ask *"how many homes did Berkeley actually build?"*, we have to get those spreadsheets into a shape a computer can reason about.

That sounds like the boring part. It's the opposite: this is where most honest mistakes are born, because **real data doesn't arrive as a tidy download — it arrives as a couple of messy spreadsheets that quietly disagree with each other.**

By the end of this notebook you'll have turned the city's raw yearly exports (2018–2025) into one clean, structured table of permits — and you'll have met the first way the raw data tries to fool you: *the file isn't even shaped like a table until you look.*

### Running the cells

To run a cell, click it and press **Shift + Return**, or click the **run (▸) button** on the cell. The simplest way through any notebook here is to start at the top and run each cell in order, reading the output that appears beneath it.

Some of the computational cells may look complex right now — that's expected, and it's fine. **You don't need to understand every line yet;** the ideas become clear as you go. Run them, watch what they produce, and keep moving.

💡 Tip: the **Next** link opens the following notebook in a new tab. If Colab says you have too many sessions, just close the previous tab and continue.

<!-- NAV:auto-generated by scripts/build_nav.py — do not edit by hand -->

← Previous: [JN0h · The agent instruction file](https://colab.research.google.com/github/blockXblock/berkeley-housing-analysis/blob/main/notebooks/curriculum/JN0h_instruction_file.ipynb)  |  Next: [JN2 · The address key](https://colab.research.google.com/github/blockXblock/berkeley-housing-analysis/blob/main/notebooks/curriculum/JN2_address_key.ipynb) →

## (run first) Colab setup

In [ ]:
# === COLAB BOOTSTRAP - fetch curriculum data + modules from R2 (NO-OP if the repo is local) ===
from pathlib import Path
import sys, urllib.request, urllib.parse, tarfile, subprocess

R2 = 'https://pub-2cee87f70da64080ab70ee0a34b55099.r2.dev/curriculum'
USE_CLEAN = False   # False: raw .xlsx path (JN1's messy-data lesson).  True (skip-ingest): permits_clean.*

_here = Path.cwd()
def _repo_ok(_here):
    """True only if a scripts/ tree exists AND housing_rules actually imports from it.
    A stale Colab extraction satisfies 'the directory exists' while being unusable, which
    previously skipped both the module refetch AND the data fetch. Anything that cannot
    import is treated as absent; under /content (a disposable Colab tree, never a real
    checkout) the broken copy is removed so the fetch below replaces it."""
    import importlib, shutil
    for _base in [_here] + list(_here.parents):
        if not (_base/'scripts'/'build_v2').exists():
            continue
        sys.path.insert(0, str(_base/'scripts'))
        try:
            for _m in [k for k in list(sys.modules)
                       if k.split('.')[0] in ('housing_rules', 's0_keys', 'cpra_dedup')]:
                del sys.modules[_m]
            importlib.invalidate_caches()
            import housing_rules  # noqa: F401  - the real test: does the package satisfy its own __init__?
            return True
        except Exception as _e:
            print(f'modules present but unusable ({type(_e).__name__}: {_e}); refetching')
            try: sys.path.remove(str(_base/'scripts'))
            except ValueError: pass
            # Remove the broken tree ONLY where it is a downloaded extraction, never a real
            # checkout: a genuine repo has .git beside scripts/. Without this removal the
            # fetch below is skipped (its own guard also only tests existence) and the stale
            # copy survives — which is precisely the bug this replaces.
            if not (_base/'.git').exists():
                shutil.rmtree(_base/'scripts', ignore_errors=True)
                print('removed the unusable scripts/ tree; it will be re-downloaded')
            return False
    return False

_have_repo = _repo_ok(_here)

def _get(url):
    # r2.dev sits behind Cloudflare, which 403s the default 'Python-urllib' User-Agent; send a browser UA.
    req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
    with urllib.request.urlopen(req, timeout=60) as r:
        return r.read()

if _have_repo:
    print('local repo detected - no fetch needed')
else:
    try:
        import pyarrow  # the parquet / USE_CLEAN path needs it; Colab has pandas, maybe not pyarrow
    except ImportError:
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pyarrow'], check=True)
    def _fetch(url, dest):
        dest = Path(dest)
        if dest.exists():
            return                                   # cached: re-runs don't re-download
        dest.parent.mkdir(parents=True, exist_ok=True)
        dest.write_bytes(_get(url)); print('fetched', dest.name)
    # 1) shared modules -> ./scripts/...  (the config-cell repo-root walk then finds scripts/build_v2)
    if not (_here/'scripts'/'build_v2').exists():
        Path('modules.tgz').write_bytes(_get(f'{R2}/curriculum_modules.tar.gz'))
        _tar = tarfile.open('modules.tgz')
        try: _tar.extractall(_here, filter='data')      # py3.12+: safe extract, no deprecation warning
        except TypeError: _tar.extractall(_here)         # older python has no filter arg
        _tar.close(); Path('modules.tgz').unlink(missing_ok=True)   # tidy: drop the intermediate tarball
        print('extracted modules -> ./scripts/')
    # 2) data -> the SAME relative paths the notebooks use (raw .xlsx AND clean exports, both fetched)
    for rel in ['data/raw/cpra-downloads/BP_Annual Permit Report-2018-2022.xlsx',
                'data/raw/cpra-downloads/BP_Annual Permit Report-2023-2025.xlsx',
                'databases/hcd_apr_mirror_2026-06-17_fresh.db',
                'databases/hcd_apr_mirror.db',
                'data/processed/permits_clean.csv',
                'data/processed/permits_clean.parquet',
                'data/processed/permits_clean_README.md']:
        _fetch(f"{R2}/data/{urllib.parse.quote(rel.split('/')[-1])}", _here/rel)   # quote -> %20 for the spaced .xlsx names
    print('curriculum bundle ready (fetched from R2)')


In [ ]:
def md(t):
    from IPython.display import Markdown, display
    display(Markdown(t))

## Config — point this at your data

In [ ]:
# === CONFIG — point this at YOUR city's permit data (this notebook is clonable) ===
from pathlib import Path
import sys, glob

# walk up to the repo root (where scripts/build_v2 lives) so the notebook runs from anywhere
REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / 'scripts' / 'build_v2').exists() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent

# --- the two knobs a student changes for another city ---
PERMIT_GLOB   = str(REPO_ROOT / 'data/raw/cpra-downloads/BP_Annual Permit Report-*.xlsx')
HEADER_ROW    = 7        # 0-indexed: Berkeley's CPRA export puts the column names on row 8
EXPECTED_UNIQUE = 30764  # the known unique-permit total for YOUR feed (Berkeley = 30,764)

# import the REAL shared modules the pipeline uses (we demonstrate them, never reinvent)
sys.path.insert(0, str(REPO_ROOT / 'scripts'))
sys.path.insert(0, str(REPO_ROOT / 'scripts' / 'build_v2'))
print('repo root :', REPO_ROOT)
print('feed files:', [Path(f).name for f in glob.glob(PERMIT_GLOB)])


## First trap: it isn't even a table yet

Here's the move everyone makes: `pd.read_excel(file)` and start computing. With civic data that's how you get silently-wrong answers, because **the file a city hands you isn't shaped like a table.** It's a document a person formatted — a title line, some blank rows — and the actual column headings sit several rows down.

**Our plan:** before trusting a single value, *look at the raw top of the file with no assumptions* — read the first few rows exactly as they are, and find where the real headings actually start.

In [ ]:
import pandas as pd
raw = pd.read_excel(glob.glob(PERMIT_GLOB)[0], dtype=str, header=None, nrows=9)
raw.iloc[:, :4]   # rows 0-7 are title/blank/metadata; the column header lives on row 7 (the 8th row)

In [ ]:
_hdr = next(i for i in range(9) if 'PermitNumber' in raw.iloc[i].astype(str).tolist())
md(f'''## What just happened

Read with *no* assumptions, the top of the file is mostly junk — a title and blank rows. The real column names (`PermitNumber`, `StreetNumber`, …) don't appear until **row {_hdr}** (counting from 0). That is why, in the next step, we load with `header={_hdr}` instead of letting pandas guess row 0.

If we'd trusted the default, every column would have been named after a scrap of the title, and every value after would have been one cell out of place — wrong in a way that still *runs*.''')

## Now stack the exports — carefully

We know where the real headings live, so we can finally read the files as tables. But "read both and stack them" hides a second trap: **the same permit can appear in two exports** — one issued in late 2022 shows up again in the 2023–2025 file — so a naive stack counts some permits twice.

**Our plan:** load each export at the correct header row, stack them into one table, drop rows that aren't real permits — and then *count* how many true, distinct permits we actually have, rather than assume.

In [ ]:
def load(path):
    d = pd.read_excel(path, dtype=str, header=HEADER_ROW)
    d.columns = [str(c).strip() for c in d.columns]
    return d

df = pd.concat([load(f) for f in glob.glob(PERMIT_GLOB)], ignore_index=True)
df = df[df['PermitNumber'].notna()].copy()    # a real permit has a number
print(f'{len(df):,} permit rows loaded')

In [ ]:
_files = len(glob.glob(PERMIT_GLOB)); _rows = len(df)
_uniq = df['PermitNumber'].nunique(); _dups = _rows - _uniq
md(f'''## What just happened

We read **{_files}** files and stacked them into **{_rows:,}** rows — but that is *not* {_rows:,} permits. Looking closer, **{_dups:,}** of those rows are the *same* permit appearing in two of the city's exports. The number of **distinct** permits is **{_uniq:,}**.

Notice we didn't *assume* there were duplicates — we counted them. A number you didn't check is a guess wearing a lab coat. (We keep every row for now; later notebooks decide what to do with the repeats.)''')

In [ ]:
import matplotlib.pyplot as plt
_yr  = df['PermitNumber'].str.extract(r'^[A-Za-z]+(\d{4})')[0]
_dup = df['PermitNumber'].duplicated(keep=False)
_tot = _yr.value_counts().sort_index()
_du  = _yr[_dup].value_counts().reindex(_tot.index, fill_value=0)
fig, ax = plt.subplots(figsize=(8, 3.2))
ax.bar(_tot.index, _tot.values, color='#cfd8dc', label='all permit rows')
ax.bar(_du.index,  _du.values,  color='#c0392b', label='same permit in two exports')
ax.set_title('Permits by the year in their permit number — and where the duplicates live')
ax.set_xlabel('year in the permit number'); ax.set_ylabel('rows'); ax.legend()
plt.xticks(rotation=45); plt.tight_layout(); plt.show()

## Keep the columns that mean one thing

A permit row has a free-text `WorkDescription` a human typed, *and* structured columns the form enforced. The temptation is to mine the description for facts. **Don't.** A fact — how many units, what date, which parcel — should come from a column with exactly one meaning, not from a sentence.

**Our plan:** select the structured columns we'll actually compute on, and keep `WorkDescription` only as *context to read*, never as a number to extract.

In [ ]:
STRUCTURED = ['PermitNumber', 'StreetNumber', 'StreetName', 'StreetType',
              'Work Type', 'UnitsAdded', 'NumberUnits', 'WorkDescription',
              'Issuance Date', 'Finaled Status', 'Finaled Date', 'Parcel Number']
feed = df[STRUCTURED].copy()
feed.head(10)

In [ ]:
md(f'''## What just happened

The raw feed has **{df.shape[1]}** columns; we kept **{len(STRUCTURED)}**. The one we deliberately keep *but will never compute on* is `WorkDescription` — useful **context**, never a **fact**. Every fact (units, dates, parcel) now comes from a structured column with one clear meaning. Parsing prose for facts is how you end up counting the word "unit" in a sentence as a home.''')

## The checkpoint: verify before you trust

We've loaded and selected — but how do we *know* it's right? Two cheap checks catch most load mistakes: the **count** of distinct permits should match the number we already know is true for this feed, and one **permit we know by heart** should read back correctly. If a header was off by a row, or a file didn't stack, one of these breaks loudly — *now*, not three notebooks later.

In [ ]:
n_unique = feed['PermitNumber'].nunique()
assert n_unique == EXPECTED_UNIQUE, f'unique permits {n_unique:,} != expected {EXPECTED_UNIQUE:,}'

b = feed[feed['PermitNumber'] == 'B2019-05574'].iloc[0]
assert b['NumberUnits'] == '135',            f"units {b['NumberUnits']}"
assert b['Work Type'] == 'New',              f"work type {b['Work Type']}"
assert str(b['Finaled Date']).startswith('2022-01-14'), f"finaled {b['Finaled Date']}"
assert b['Parcel Number'] == '055 189501805', f"apn {b['Parcel Number']}"

print(f'CHECKPOINT PASS')
print(f'  {n_unique:,} unique permits (== known feed total)')
print(f'  B2019-05574: 135u / New / Finaled 2022-01-14 / APN 055 189501805')

In [ ]:
md(f'''## What just happened

Both checks passed. There are exactly **{n_unique:,}** distinct permits — the number we expected — so the stack-and-load didn't silently drop or double anything. And our known permit **B2019-05574** reads back as 135 units, a *New* building, finaled 2022-01-14, on parcel 055 189501805 — every field where we expected it.

That's the whole habit of this course in one cell: **state what should be true, then make the computer prove it.** A green check you can re-run beats a number you remember being right.''')

**JN1 done.** You turned a hand-formatted document into a structured, verified table of permits — and you already met the first lie raw data tells: *it looks like a table before it is one.* **Next — JN2:** the address key, the quiet foundation everything else stands on, and the trap where the *same place* wears two different names.

<!-- NAV:auto-generated by scripts/build_nav.py — do not edit by hand -->

← Previous: [JN0h · The agent instruction file](https://colab.research.google.com/github/blockXblock/berkeley-housing-analysis/blob/main/notebooks/curriculum/JN0h_instruction_file.ipynb)  |  Next: [JN2 · The address key](https://colab.research.google.com/github/blockXblock/berkeley-housing-analysis/blob/main/notebooks/curriculum/JN2_address_key.ipynb) →